# HeliciteCinetique — `helicite_cinetique`

`Postraitement_fields` · ASN 28 validation form (TRUST-LiMe).


## Introduction

Validation field of the **standalone magnetic (induction) equation** solved by
TRUST-LiMe:

$$\partial_t b + (u \cdot \nabla) b - \nabla \cdot (\eta \nabla b) = 0$$

This notebook runs **one and only one test case**: ``helicite_cinetique`. Per the TRUST validation-form layout
(`Validation/Rapports_automatiques/<Category>/<FormName>/`), the other
`Postraitement_fields` cases live in their own sibling folders.

## What it validates

Kinetic helicity density built from the element gradient of the frozen
velocity. 3D linear velocity $u = (y, z, x)$: $\nabla \times u = -(1,1,1)$
exactly, so $h_k = -(x + y + z)$. The probe sits at the cell centre
$(0.375, 0.375, 0.375)$ where $h_k = -1.125$. (In 2D $h_k \equiv 0$:
out-of-plane vorticity, in-plane velocity.)

## Contract checked

| # | Quantity | Formula |
|---|---|---|
| 1 | kinetic helicity density | $h_k = u \cdot (\nabla \times u)$ |

Computed in the module by the `Champ_Fonc_MHD` post-treatment field
(`src/Champ_Fonc_MHD.{h,cpp}` + `src/MHD_posttraitement_kernel.h`).


In [ ]:
from trustutils import run

run.TRUST_parameters("1.9.8")
run.reset()


## Helpers: read a TRUST probe (`.son`) file

The case is copied by `trustutils` into `run.BUILD_DIRECTORY`; the `.son`
probe files are written next to the dataset in `src/` and copied over in the
run cell below, where `read_son()` expects them.


In [ ]:
import numpy as np
from trustutils.files import SonFile

BUILD = run.BUILD_DIRECTORY

def read_son(case, sonde):
    """Return (time, values) for probe `sonde` of case `case` (dataset name)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    entries = son.getEntries()
    assert entries, f"no probe entry found in {path}"
    t, v = son.getValues(entries[0])
    return np.asarray(t, dtype=float), np.asarray(v, dtype=float)

def read_son_all(case, sonde):
    """Like read_son but returns every entry (components of a vector field)."""
    path = f"{BUILD}/{case}_{sonde}.son"
    son = SonFile(path, None)
    return [(e, np.asarray(son.getValues(e)[1], dtype=float)) for e in son.getEntries()]

def final_value(case, sonde):
    """Field value at the probe, at the last recorded time."""
    t, v = read_son(case, sonde)
    # the first line (t=0) is written before the problem is initialised
    return t, v[-1]

print("BUILD =", BUILD)


## Run

Sequential-only: the dataset is run by `trustutils` on one processor
(no `partition()`, no `nbProcs`).


In [ ]:
import pathlib, shutil

from trustutils import run

run.reset()
run.addCase(".", "helicite_cinetique.data")      # sequential-only: never nbProcs > 1
run.printCases()
run.runCases()

# trustutils runs the case in its build tree; the .son probes are written next
# to the dataset in src/, so copy them where read_son() looks for them.
build = pathlib.Path(run.BUILD_DIRECTORY)
build.mkdir(parents=True, exist_ok=True)
for son in pathlib.Path("src").glob("helicite_cinetique_*.son"):
    shutil.copy2(son, build / son.name)


## Checks


In [ ]:
t, hk = final_value("helicite_cinetique", "SONDE_HK")

expected = -(0.375 + 0.375 + 0.375)
print(f"final time                : {t[-1]:.6g}")
print(f"helicite_cinetique (probe): {hk:.6g}  (expected {expected:.6g})")
assert abs(hk - expected) < 1e-8, f"helicite_cinetique wrong: {hk}"
print("OK: helicite_cinetique = u . curl(u).")


## Computer Performance


In [ ]:
run.tablePerf()
